# Sevastopol AI — парсер каналов (Google Colab, Telethon)

Читает любые публичные каналы с **личного аккаунта** и копирует новые посты в Избранное.

1. Ячейка 1 — установка telethon.
2. Ячейка 2 — вход: спросит телефон и код из Telegram, напечатает строку сессии.
3. Ячейка 3 — вставь SESSION из ячейки 2 и список каналов.
4. Ячейка 4 — запуск (работает, пока не остановишь ячейку).

⚠️ **SESSION = полный доступ к аккаунту.** Не показывай ноутбук посторонним,
не публикуй его. Стейт лежит в `/content/forwarder_state.json` — дубликатов после
перезапуска ячейки не будет.

In [ ]:
!pip install -q telethon

In [ ]:
from telethon import TelegramClient
from telethon.sessions import StringSession

API_ID = int(input("TELETHON_API_ID (my.telegram.org → API development tools): ").strip())
API_HASH = input("TELETHON_API_HASH: ").strip()

client = TelegramClient(StringSession(), API_ID, API_HASH)
client.parse_mode = None
await client.start()              # спросит телефон, код из Telegram, пароль 2FA
me = await client.get_me()
print("Аккаунт:", me.first_name, "@%s" % me.username, "id", me.id)
print("\nTELETHON_SESSION (скопируй в ячейку 3):")
print(client.session.save())
await client.disconnect()

In [ ]:
API_ID = 00000000                                        # ← свой: my.telegram.org → API development tools
API_HASH = "00000000000000000000000000000000"            # ← свой оттуда же (или из events/.env)
SESSION = ""                                             # ← строка из ячейки 2
CHANNELS = ["https://t.me/Sevastopol_AI"]                # любые публичные каналы
TARGET = "me"                                            # "me" = Избранное
POLL_SECONDS = 12
INCLUDE_HISTORY = False

In [ ]:
import asyncio, io, json, os, time
from telethon import TelegramClient
from telethon.errors import FloodWaitError
from telethon.sessions import StringSession

STATE_PATH = "/content/forwarder_state.json"
FETCH_LIMIT, TEXT_LIMIT, CAPTION_LIMIT = 15, 4096, 1024
NON_FILE_MEDIA = ("MessageMediaWebPage", "MessageMediaContact", "MessageMediaGeo",
                  "MessageMediaGeoLive", "MessageMediaVenue", "MessageMediaPoll",
                  "MessageMediaDice", "MessageMediaInvoice", "MessageMediaGame",
                  "MessageMediaStory")

def log(m): print(time.strftime("[%H:%M:%S] ") + m, flush=True)

def norm_source(s):
    s = str(s or "").strip()
    for p in ("https://", "http://"):
        if s.lower().startswith(p): s = s[len(p):]
    for d in ("t.me/", "telegram.me/", "telegram.dog/"):
        if s.lower().startswith(d): s = s[len(d):]
    s = s.split("?", 1)[0].split("#", 1)[0].strip("/")
    parts = [p for p in s.split("/") if p]
    if not parts: return ""
    if parts[0].lower() == "c" and len(parts) > 1 and parts[1].lstrip("-").isdigit():
        return "-100" + parts[1].lstrip("-")
    if parts[0].lower() == "s" and len(parts) > 1: parts = parts[1:]
    name = parts[0].lstrip("@").strip()
    return "" if (not name or name.startswith("+")) else name

def post_link(src, mid):
    src = str(src)
    if src.lstrip("-").isdigit():
        internal = src[4:] if src.startswith("-100") else src.lstrip("-")
        return "https://t.me/c/%s/%s" % (internal, mid)
    return "https://t.me/%s/%s" % (src.lstrip("@"), mid)

def group_new(msgs):
    groups, index = [], {}
    for m in msgs:
        gid = getattr(m, "grouped_id", None)
        if gid:
            if gid in index: index[gid].append(m)
            else:
                b = [m]; index[gid] = b; groups.append(b)
        else: groups.append([m])
    return groups

def clip_entities(ents, limit):
    if not ents: return None
    out = []
    for e in ents:
        o, l = getattr(e, "offset", 0), getattr(e, "length", 0)
        if o >= limit: continue
        if o + l > limit:
            try: e = e.clone()
            except Exception: continue
            e.length = limit - o
        out.append(e)
    return out or None

def media_is_file(m):
    media = getattr(m, "media", None)
    return media is not None and type(media).__name__ not in NON_FILE_MEDIA

def load_state():
    try:
        with open(STATE_PATH, encoding="utf-8") as f:
            d = json.load(f)
        if isinstance(d, dict) and isinstance(d.get("channels"), dict): return d
    except Exception: pass
    return {"version": 2, "channels": {}}

def save_state(state):                       # атомарно: tmp + os.replace
    state["updated"] = time.strftime("%Y-%m-%d %H:%M:%S")
    tmp = STATE_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f: json.dump(state, f, ensure_ascii=False, indent=2)
    os.replace(tmp, STATE_PATH)

async def send_media(client, target, m, caption, ents):
    try:
        await client.send_file(target, file=m.media, caption=caption,
                               formatting_entities=ents,
                               attributes=getattr(m, "attributes", None))
        return
    except Exception as e:
        log("   ⚠️ медиа по ссылке не ушло (%s) — скачиваю" % e)
    buf = io.BytesIO(); await client.download_media(m, file=buf); buf.seek(0)
    await client.send_file(target, file=buf, caption=caption, formatting_entities=ents)

async def copy_group(client, target, key, entity, group, title):
    ids = [m.id for m in group]
    link = post_link(key, ids[0])
    try:                                     # 1) форвард: альбом едет группой
        await client.forward_messages(target, ids, from_peer=entity)
        log("   ↪️ форвард «%s» id %s" % (title, ids if len(ids) > 1 else ids[0]))
        return
    except FloodWaitError as e:
        log("   ⏳ FloodWait %d c" % e.seconds); await asyncio.sleep(e.seconds)
        await client.forward_messages(target, ids, from_peer=entity); return
    except Exception as e:                   # 2) запрет форвардов → ручная копия
        log("   ⚠️ форвард не сработал (%s) — копирую вручную" % e)
    album = len(group) > 1
    for i, m in enumerate(group):
        text = m.message or ""
        tail_link = link if (not album or i == len(group) - 1) else None
        tail = "\n\n🔗 " + tail_link if tail_link else ""
        try:
            if media_is_file(m):
                if len(text) + len(tail) > CAPTION_LIMIT:
                    await send_media(client, target, m, text[:CAPTION_LIMIT],
                                     clip_entities(m.entities, CAPTION_LIMIT))
                    if tail_link: await client.send_message(target, "🔗 " + tail_link)
                else:
                    await send_media(client, target, m, text + tail, m.entities)
            else:
                body = text or ("(медиа без описания)" if getattr(m, "media", None) else "")
                if len(body) + len(tail) <= TEXT_LIMIT:
                    await client.send_message(target, body + tail, formatting_entities=m.entities)
                else:
                    await client.send_message(target, body[:TEXT_LIMIT],
                                              formatting_entities=clip_entities(m.entities, TEXT_LIMIT))
                    if tail_link: await client.send_message(target, "🔗 " + tail_link)
            log("   📄 копия «%s» id %d" % (title, m.id))
        except FloodWaitError as e:
            log("   ⏳ FloodWait %d c" % e.seconds); await asyncio.sleep(e.seconds)
        except Exception as e:
            log("   ⚠️ не скопировал id %d: %s" % (m.id, e))
            try: await client.send_message(target, "🔗 " + link)
            except Exception: pass

async def poll_channel(client, cfg_target, src, state):
    key, entity, title = src["key"], src["entity"], src["title"]
    try:
        msgs = list(await client.get_messages(entity, limit=FETCH_LIMIT) or [])
    except FloodWaitError as e:
        log("⏳ FloodWait «%s»: %d c" % (title, e.seconds)); await asyncio.sleep(e.seconds); return 0
    except Exception as e:
        log("⚠️ «%s»: %s" % (title, e)); return 0
    if not msgs: return 0
    asc = sorted(msgs, key=lambda m: m.id)
    newest = asc[-1].id
    saved = state["channels"].get(key)
    if saved is None:                        # первый запуск
        saved = max(0, newest - FETCH_LIMIT) if INCLUDE_HISTORY else newest
        log("«%s»: отсчёт с id %s%s" % (title, saved,
            "" if INCLUDE_HISTORY else " (историю не копирую)"))
    fresh = [m for m in asc if m.id > saved and not getattr(m, "action", None)]
    if fresh:
        log("📨 «%s»: новых %d" % (title, len(fresh)))
        for g in group_new(fresh):
            try:
                await copy_group(client, cfg_target, key, entity, g, title)
            except FloodWaitError as e:
                log("⏳ FloodWait %d c" % e.seconds); await asyncio.sleep(e.seconds)
                try: await copy_group(client, cfg_target, key, entity, g, title)
                except Exception as e2: log("   ⚠️ %s" % e2)
            except Exception as e:
                log("⚠️ ошибка на id %s: %s" % ([m.id for m in g], e))
    state["channels"][key] = max(newest, state["channels"].get(key) or 0)
    save_state(state)
    return len(fresh)

client = TelegramClient(StringSession(SESSION.strip()), API_ID, API_HASH.strip())
client.parse_mode = None                    # шлём текст как есть + исходные entities
await client.connect()
if not await client.is_user_authorized():
    raise SystemExit("Сессия недействительна — выполни ячейку 2 заново")
me = await client.get_me()
log("Аккаунт: %s (@%s)" % (me.first_name, me.username))
target = "me" if str(TARGET).lower() in ("me", "self", "saved") else (
    int(TARGET) if str(TARGET).lstrip("-").isdigit() else norm_source(TARGET))
sources = []
for raw in CHANNELS:
    key = norm_source(raw)
    if not key: continue
    try:
        e = await client.get_entity(int(key) if key.lstrip("-").isdigit() else key)
        sources.append({"key": key, "entity": e, "title": getattr(e, "title", None) or key})
    except Exception as ex:
        log("⚠️ канал %r недоступен: %s" % (raw, ex))
if not sources: raise SystemExit("Ни один канал не открылся — проверь ссылки")
state = load_state()
log("Каналов: %d · опрос раз в %d c · цель: %s" % (len(sources), POLL_SECONDS, TARGET))
try:
    while True:
        for src in sources:
            await poll_channel(client, target, src, state)
        await asyncio.sleep(POLL_SECONDS)
except KeyboardInterrupt:
    log("Остановлено. Прогресс в %s" % STATE_PATH)
finally:
    save_state(state)
    await client.disconnect()